# Adding Chat: Memory + Query Condensation

`02-Basic_RAG_Formulary.ipynb`'s `ask()` is stateless — every call is a one-off, with no memory of what was asked before. Real usage isn't like that: a member naturally asks follow-ups (*"What tier is atorvastatin?"* → *"What about quantity limits?"*). This notebook follows `ai-tutor-rag-system/notebooks/14-Adding_Chat.ipynb` to add that — but adapts which pieces actually apply:

- **Chat memory** — adopted. Needed for any follow-up to make sense at all.
- **Condense-question (rewrite a follow-up into a standalone question using history, *before* retrieving)** — adopted, and load-bearing, not cosmetic. Retrieval here is hybrid dense+BM25 (`05-Hybrid_Search_Formulary.ipynb`). BM25 matches literal words — a follow-up like *"Does it have any quantity limits?"* contains **no drug name token at all**, so BM25 finds nothing and dense search has little to anchor on either. Adding memory without condensing first would make follow-ups retrieve *worse*, not better. Section 5 below proves this with a real before/after, the same way `05` proved hybrid search with real numbers rather than asserting it.
- **Streaming** — adopted. `requirements.txt` already includes Gradio, so a real chat UI is clearly the direction, and local Ollama generation is slow enough that showing tokens as they arrive matters more here than with a cloud API.
- **ReAct agent** — left out. Agentic tool-calling earns its keep with multiple tools to choose between (see the `13-Adding_Router.ipynb` discussion); there's still exactly one retriever here.

## 1. Setup

In [1]:
import yaml
from pathlib import Path

config_path = Path("../../config/config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print(f"✅ Config loaded. Configured top_k: {config['retrieval']['top_k']}")

✅ Config loaded. Configured top_k: 3


## 2. Load the Cached Vector Store + Hybrid Retriever

Same reuse principle as `04`/`05`: no PDF re-parse, no re-embedding. Rows for both the FAISS and BM25 halves come straight out of the existing cache's docstore.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

embeddings = HuggingFaceEmbeddings(model_name=config["embedding"]["model_name"])

MODEL_TAG = config["embedding"]["model_name"].replace("/", "-")
CACHE_DIR = Path(f"../../{config['paths']['processed_data']}/faiss_cache_{MODEL_TAG}_{config['embedding']['dimension']}d")
INDEX_NAME = "formulary_rows"

assert (CACHE_DIR / f"{INDEX_NAME}.faiss").exists(), (
    f"No cache at {CACHE_DIR} — run 02-Basic_RAG_Formulary.ipynb first to build it."
)

vectorstore = FAISS.load_local(
    str(CACHE_DIR), embeddings, index_name=INDEX_NAME, allow_dangerous_deserialization=True
)
all_docs = list(vectorstore.docstore._dict.values())

bm25_retriever = BM25Retriever.from_documents(all_docs)


def build_hybrid_retriever(k, vector_weight=0.5, bm25_weight=0.5):
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    bm25_retriever.k = k
    return EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[vector_weight, bm25_weight],
    )


print(f"✅ Loaded {len(all_docs)} rows; hybrid retriever ready")

/tmp/ipykernel_28834/3097454301.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


/home/bhargav/portfolio-project/healthins-enrollment-assistant-dev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3829.87it/s]

✅ Loaded 2394 rows; hybrid retriever ready


## 3. LLM + Prompting (same as `02`)

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=config["llm"]["model"],
    temperature=config["llm"]["temperature"],
    num_predict=config["llm"]["max_tokens"],
)

SYSTEM_INSTRUCTION = (
    "You are a helpful assistant answering questions about health insurance plans. "
    "Use only the provided context to answer. If the context doesn't contain the answer, say you don't know."
)

PROMPT_TEMPLATE = (
    "Read the following informations that might contain the context you require to "
    "answer the question. You can use the informations starting from the "
    "<START_OF_CONTEXT> tag and end with the <END_OF_CONTEXT> tag. Here is the content:\n\n"
    "<START_OF_CONTEXT>\n{context}\n<END_OF_CONTEXT>\n\n"
    "Please provide an informative and accurate answer to the following question based "
    "on the available context. Be concise and take your time.\nQuestion: {question}\nAnswer:"
)


def format_sources(docs):
    seen = set()
    lines = []
    for doc in docs:
        drug = doc.metadata.get("drug", "unknown")
        if drug in seen:
            continue
        seen.add(drug)
        page = doc.metadata.get("page", "?")
        section = doc.metadata.get("section") or "N/A"
        lines.append(f"- {drug} — p.{page}, {section}")
    return "\n".join(lines)


print(f"✅ LLM ready: {config['llm']['model']}")

✅ LLM ready: llama3.1


## 4. Condense-Question + Chat Memory

`condense_question` only calls the LLM when there *is* history — the first turn of any conversation is passed straight through, so a one-off question costs exactly what it costs today. `chat()` keeps a plain Python list as memory (no framework needed for this), condenses the question, retrieves with the hybrid retriever, and streams the answer.

In [4]:
CONDENSE_PROMPT = (
    "Given the conversation history and a follow-up question, rewrite the follow-up "
    "question to be a standalone question that includes any necessary context from the "
    "history (such as the drug name being discussed). If the follow-up question is "
    "already standalone, return it unchanged. Output only the rewritten question, "
    "nothing else — no preamble, no quotes.\n\n"
    "Chat History:\n{history}\n\n"
    "Follow-up Question: {question}\n"
    "Standalone Question:"
)

chat_history = []  # list of (question, answer) tuples


def format_history(history, max_turns=3):
    return "\n".join(f"User: {q}\nAssistant: {a}" for q, a in history[-max_turns:])


def condense_question(question, history, llm):
    if not history:
        return question
    prompt = CONDENSE_PROMPT.format(history=format_history(history), question=question)
    response = llm.invoke([{"role": "user", "content": prompt}])
    return response.content.strip()


def chat(question, k=None, stream=True):
    k = k or config["retrieval"]["top_k"]

    standalone_question = condense_question(question, chat_history, llm)
    if standalone_question != question:
        print(f'(condensed to: "{standalone_question}")\n')

    retriever = build_hybrid_retriever(k)
    docs = retriever.invoke(standalone_question)[:k]
    context = "\n\n".join(d.page_content for d in docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=standalone_question)
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": prompt},
    ]

    if stream:
        full_answer = ""
        for chunk in llm.stream(messages):
            print(chunk.content, end="", flush=True)
            full_answer += chunk.content
        print()
    else:
        full_answer = llm.invoke(messages).content

    chat_history.append((question, full_answer))
    answer_with_sources = f"{full_answer}\n\nSources:\n{format_sources(docs)}"
    return answer_with_sources, docs


def reset_chat():
    chat_history.clear()


print("✅ Chat session ready")

✅ Chat session ready


## 5. Proof: Does Condensation Actually Rescue a Follow-Up?

Same standard as `05`'s hybrid-search comparison — measure it, don't assert it. A two-turn conversation about a real drug in this formulary, where turn 2 deliberately contains no drug name at all.

In [5]:
reset_chat()

turn1 = "What tier is atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg?"
print("=== Turn 1 ===")
print(turn1, "\n")
answer1, docs1 = chat(turn1, stream=False)
print("\n" + answer1)

=== Turn 1 ===
What tier is atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg? 




According to the provided context, the tier for atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg is Tier 1.

Sources:
- amlodipine besylate-atorvastatin calcium tab 10-80 mg — p.42, CALCIUM CHANNEL BLOCKER/ANTILIPEMIC COMBINATIONS
- atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg — p.39, ANTILIPEMICS, HMG-COA REDUCTASE INHIBITORS
- amlodipine besylate-atorvastatin calcium tab 10-20 mg — p.42, CALCIUM CHANNEL BLOCKER/ANTILIPEMIC COMBINATIONS


In [6]:
follow_up = "Does it have any quantity limits or other requirements?"
k = config["retrieval"]["top_k"]

print("=== WITHOUT condensation: retrieving on the raw follow-up text ===")
raw_docs = build_hybrid_retriever(k).invoke(follow_up)
for d in raw_docs:
    print("-", d.metadata["drug"], "| p.", d.metadata["page"])

print("\n=== WITH condensation ===")
standalone = condense_question(follow_up, chat_history, llm)
print("Condensed question:", standalone, "\n")
condensed_docs = build_hybrid_retriever(k).invoke(standalone)
for d in condensed_docs:
    print("-", d.metadata["drug"], "| p.", d.metadata["page"])

=== WITHOUT condensation: retrieving on the raw follow-up text ===
- desmopressin acetate spray refrigerated soln .01% | p. 73
- goodsense aspirin chew 81mg | p. 16
- QUADRAMET SOLN 1850MBQ/ML | p. 33
- aspirin ec adult low dose tbec 81mg | p. 16
- desmopressin acetate spray soln .01% | p. 73
- ed at the p | p. 7

=== WITH condensation ===


Condensed question: Does atorvastatin calcium have any quantity limits or other requirements? 

- amlodipine besylate-atorvastatin calcium tab 10-80 mg | p. 42
- atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg | p. 39
- amlodipine besylate-atorvastatin calcium tab 2.5-10 mg | p. 42
- acamprosate calcium tbec 333mg | p. 46
- amlodipine besylate-atorvastatin calcium tab 5- 80 mg | p. 42
- amlodipine besylate-atorvastatin calcium tab 2.5-40 mg | p. 42


In [7]:
print("=== Turn 2 (streamed, through the real chat() function) ===")
print(follow_up, "\n")
answer2, docs2 = chat(follow_up)
print("\n" + answer2)

=== Turn 2 (streamed, through the real chat() function) ===
Does it have any quantity limits or other requirements? 



(condensed to: "Does atorvastatin calcium have any quantity limits or other requirements?")



According

 to

 the

...

 quantity

 limits

 mentioned

.



According to the context, atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg have an exception process available for a $0 copay for members age 40 through 75 when medically necessary for primary prevention of cardiovascular disease. However, there are no quantity limits mentioned.

Sources:
- amlodipine besylate-atorvastatin calcium tab 10-80 mg — p.42, CALCIUM CHANNEL BLOCKER/ANTILIPEMIC COMBINATIONS
- atorvastatin calcium tabs 10mg, 20mg, 40mg, 80mg — p.39, ANTILIPEMICS, HMG-COA REDUCTASE INHIBITORS
- amlodipine besylate-atorvastatin calcium tab 2.5-10 mg — p.42, CALCIUM CHANNEL BLOCKER/ANTILIPEMIC COMBINATIONS


## 6. Reset Clears Memory

Asking the same pronoun-style follow-up right after `reset_chat()` should fail again — memory, not luck, is what made turn 2 above work.

In [8]:
reset_chat()
answer3, docs3 = chat(follow_up, stream=False)
print(answer3)

According to the context, there are quantity limits or requirements for the following drugs:

* Goodsense Aspirin Chew 81mg: QL (100 tabs every 30 days)
* QUADRAMET SOLN 1850MBQ/ML: None
* desmopressin acetate spray refrigerated soln .01%: None

So, the answer is that there are quantity limits or requirements for Goodsense Aspirin Chew 81mg, but not for the other two drugs.

Sources:
- desmopressin acetate spray refrigerated soln .01% — p.73, VASOPRESSINS
- goodsense aspirin chew 81mg — p.16, SALICYLATES
- QUADRAMET SOLN 1850MBQ/ML — p.33, MISCELLANEOUS


## 🔑 Key Takeaways

- **Condensation isn't decorative here — it's the only thing standing between memory and broken retrieval.** BM25 needs literal words; a pronoun-only follow-up gives it none. Section 5's before/after shows the raw follow-up landing on unrelated rows, and the condensed version landing back on the correct one.
- **Only pay the extra LLM call when there's actually history to use.** `condense_question` short-circuits on the first turn of any conversation — a one-off question isn't taxed for a feature it doesn't need.
- **Streaming matters more with a local model than a cloud one.** Ollama generation is slow enough that perceived latency (tokens appearing immediately) is a real UX difference, not polish.
- **Memory is a plain Python list, not a framework dependency.** No new library was needed to keep `(question, answer)` turns around — LangChain's fancier memory abstractions would be solving a problem this project doesn't have yet.
- **The agent/ReAct piece from `14` was left out on purpose.** Same reasoning as skipping `13`'s router: it's justified by multiple tools to orchestrate, and there's still exactly one retriever here.